In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "silver")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable

In [ ]:

def merge_to_silver(table_name, primary_key):
    
    df_bronze = spark.table(f"{catalog}.bronze.{table_name}")
    
    # 2. Clean it
    df_cleaned = df_bronze \
        .dropDuplicates([primary_key]) \
        .dropna(subset=[primary_key]) \
        .drop("ingestion_date", "source_path")
    
    # 3. Check if silver table exists
    full_table_name = f"{catalog}.silver.{table_name}_cleaned"
    
    if spark.catalog.tableExists(full_table_name):
        # TABLE EXISTS → do MERGE (upsert)
        delta_table = DeltaTable.forName(spark, full_table_name)
        
        delta_table.alias("target").merge(
            df_cleaned.alias("source"),
            f"target.{primary_key} = source.{primary_key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print(f"✅ MERGED into {full_table_name}")
    
    else:
        # TABLE DOESN'T EXIST → do full load
        df_cleaned.write.mode("overwrite") \
            .saveAsTable(full_table_name)
        
        print(f"✅ CREATED {full_table_name}")

In [ ]:

merge_to_silver("customers",    primary_key="customer_id")
merge_to_silver("products",     primary_key="product_id")
merge_to_silver("orders",       primary_key="order_id")
merge_to_silver("order_items",  primary_key="order_item_id")